# <span style="color:darkblue"> Lecture 10c: Analyzing Experiments </span>

<font size = "5">



# <span style="color:darkblue"> I. Import Libraries </span>


<font size = "5">

Install `maketables` in your notebook environment with `pip install maketables` if it is not already installed.

In [1]:
# The "pandas" library is used for processing datasets
# The "numpy" is for numeric observations and random numbers
# The "matplotlib.pyplot" library is for creating graphs

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# We will "alias" two sublibraries in "statsmodels"
# "statsmodels.formula.api" contains functions to estimate models
# "statsmodels.api" contains general-use statistical options

import statsmodels.api as sm
import statsmodels.formula.api as smf
import maketables as mt

<font size = "5">

Import data

In [2]:
dataset = pd.read_stata("data_raw/malawiexperiment.dta")

# <span style="color:darkblue"> II. Context </span>


<font size = "5">

Today we will review a paper by Rebecca Dizon-Ross published <br>
in the American Economic Review (2019).

- In this study, researchers partnered with local schools in Malawi <br>
- This study evaluated the impacts of information about children's <br>
 academic performance on parents subsequent investments in their <br>
  children's education.


https://www.povertyactionlab.org/evaluation/effects-student-performance-information-parental-decision-making-malawi?lang=fr

https://www.nber.org/papers/w24610

<font size = "5">

Intervention

- Parents in Malawi with low literacy levels had trouble interpreting <br>
school report cards.Many parents were unaware that their children were <br>
struggling with school.
- The intervention altered the way that the schools engaged with <br>
 parents. It had an impact on reducing the information gaps.


<font size = "5">

Experimental Design

- Students were assigned to randomly to treatment and control <br>
with 50% probability
- The random assignment was done at the household level

<font size = "5">

Findings


<img src="figures/treatmenteffects_dizonross.png" alt="drawing" width="650"/>



<font size = "5">

At baseline (before the intervention)

- The graph on the left shows parental beliefs at baseline
- Parents of low performing students thought that they were doing <br>
better than they were. Ideally it should  be along the 45-degree line
- Similar results for treated and control groups at baseline because <br>
of randomization

At endline (after the intervention)

- Treated parents had more accurate perceptions of their children's <br>
performance
- The treatment effects varied depending on the baseline test scores


# <span style="color:darkblue"> II. Basic Descriptive Analysis </span>


<font size = "5">

Total number of children

In [3]:
len(dataset)

5268

<font size = "5">

Total number of households

- 2 children per households

In [4]:
unique_ids = pd.unique(dataset["hhid"])
len(unique_ids)

2634

<font size = "5">

Calculate number of treated and control

In [5]:
table = pd.crosstab(index = dataset['treat'],columns = "count")
table

col_0,count
treat,
Control,2654
Treatment,2614


# <span style="color:darkblue"> III. Testing Covariate Balance </span>


<font size = "5">

Subset treated and control observations

In [6]:
dataset_treated = dataset.query('treat == "Treatment"')
dataset_control = dataset.query('treat == "Control"')

<font size = "5">

Socio-economic information can be collected at baseline <br>
(before the experiment) 

In [7]:
variables_scores      = ["ave"]
variables_respondent  = ["lit","primary_resp_fem","age_par1","farmer"]
variables_household   = ["tot_kids","one_par"]
variables_student     = ["std","age","female","attendance_sv"]

<font size = "5" >

Check that characteristics are similar between treated and control <br>
at baseline

In [8]:
# Compute mean and standard deviation for the treated group
display(dataset_treated[variables_respondent].describe().loc[['mean', 'std']])

# Compute mean and standard deviation for the control group
display(dataset_control[variables_respondent].describe().loc[['mean', 'std']])

,lit,primary_resp_fem,age_par1,farmer
mean,0.675613,0.758990,40.974080,0.460587
std,0.468235,0.427778,11.290328,0.498541


,lit,primary_resp_fem,age_par1,farmer
mean,0.667426,0.773926,40.645455,0.465544
std,0.471225,0.418366,10.638597,0.498907


<font size = "5">



<font size = "5" >

Conduct a formal test of whether the coefficients are similar

- We should expect the coefficient on the treatment variable <br>
to be non-significant

In [9]:
reg_model = smf.ols("lit ~ treat ", dataset)
results = reg_model.fit(cov_type= "HC1")

mt.ETable([results], model_heads=["Baseline balance"]).make(type="gt")

GT(_tbl_data=  __index_level_0__ __index_level_1__                      0
0              coef   treat=Treatment     0.008 <br> (0.013)
1              coef         Intercept  0.667*** <br> (0.009)
2             stats      Observations                  5,242
3             stats                R²               0.000076, _body=<great_tables._gt_data.Body object at 0x31a42ee40>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x35e7a7610>, _spanners=Spanners([SpannerInfo(spanner_id='lit', spanner_level=2, spanner_label=Html(text='lit'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='Baseline balance', spanner_level=1, spanner_label=Html(text='Baseline balance'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x31a42f380>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x35e7a60d0>, _source_notes=[Html(text='Significance levels: * p &lt; 0.1, ** p &lt; 0.05, *** p &lt; 0.01. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x31a42f4d0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x31a42f8c0>, <great_tables._gt_data.FormatInfo object at 0x35e7a6e90>, <great_tables._gt_data.FormatInfo object at 0x35e7a7250>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=Options

<font size = "5">

Try it yourself!

<font size = "3">

- Obtain summary statistics for the mean and standard deviation for <br>
the other sets of baseline variables


In [10]:
# Write your own code

display(dataset[variables_respondent].describe().loc[["mean", "std"]])
display(dataset[variables_household].describe().loc[["mean", "std"]])
display(dataset[variables_student].describe().loc[["mean", "std"]])




,lit,primary_resp_fem,age_par1,farmer
mean,0.671499,0.766515,40.808811,0.463077
std,0.469713,0.423088,10.967499,0.498683


,tot_kids,one_par
mean,5.132878,0.192863
std,1.736976,0.394584


,std,age,female,attendance_sv
mean,3.720577,11.617009,0.508058,0.914661
std,1.365353,2.675271,0.499984,0.128882


<font size = "5">

Try it yourself!

<font size = "3">

- Write a loop that runs different regressions of baseline covariates <br>
on the treatment variable. This can help you automate the process of <br>
testing for covariate balance

In [12]:
# Write your own code



baseline_variables = (
    variables_scores
    + variables_respondent
    + variables_household
    + variables_student
)

results_list = []

for variable in baseline_variables:
    reg_model = smf.ols(f"{variable} ~ treat", data=dataset)
    results = reg_model.fit(cov_type="HC1")
    results_list.append(results)

# Use unique model headings
model_heads = [f"Balance test: {v}" for v in baseline_variables]

mt.ETable(
    results_list,
    model_heads=model_heads
).make(type="gt")




GT(_tbl_data=  __index_level_0__ __index_level_1__                       0  \
0              coef   treat=Treatment     -0.773 <br> (0.482)   
1              coef         Intercept  47.131*** <br> (0.339)   
2             stats      Observations                   5,268   
3             stats                R²                0.000489   

                       1                      2                       3  \
0     0.008 <br> (0.013)    -0.015 <br> (0.012)      0.329 <br> (0.316)   
1  0.667*** <br> (0.009)  0.774*** <br> (0.008)  40.645*** <br> (0.216)   
2                  5,242                  5,268                   4,812   
3               0.000076               0.000312                0.000224   

                       4                      5                      6  \
0    -0.005 <br> (0.014)    -0.062 <br> (0.048)     0.012 <br> (0.011)   
1  0.466*** <br> (0.010)  5.164*** <br> (0.034)  0.187*** <br> (0.008)   
2                  5,200                  5,268                  5,268   
3               0.000025               0.000316               0.000233   

                       7                       8                      9  \
0     0.005 <br> (0.038)     -0.096 <br> (0.076)    -0.018 <br> (0.014)   
1  3.718*** <br> (0.026)  11.665*** <br> (0.053)  0.517*** <br> (0.010)   
2                  5,268                   5,021                  5,088   
3               0.000003                0.000324               0.000309   

                      10  
0    -0.002 <br> (0.004)  
1  0.915*** <br> (0.003)  
2                  5,228  
3               0.000041  , _body=<great_tables._gt_data.Body object at 0x35e640e90>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(2)'), column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(3)'), column_align='center', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(4)'), column_align='center', column_width=None), ColInfo(var='4', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(5)'), column_align='center', column_width=None), ColInfo(var='5', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(6)'), column_align='center', column_width=None), ColInfo(var='6', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(7)'), column_align='center', column_width=None), ColInfo(var='7', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(8)'), column_align='center', column_width=None), ColInfo(var='8', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(9)'), column_align='center', column_width=None), ColInfo(var='9', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(10)'), column_align='center', column_width=None), ColInfo(var='10', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(11)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x31a4045f0>, _spanners=Spanners([SpannerInfo(spanner_id='ave', spanner_level=2, spanner_label=Html(text='ave'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='lit', spanner_level=2, spanner_label=Html(text='lit'), spanner_units=None, spanner_pattern=None, vars=['1'], built=None), SpannerInfo(spanner_id='primary_resp_fem', spanner_level=2, spanner_label=Html(text='primary_resp_fem'), spanner_units=None, spanner_pattern=None, vars=['2'], built=None), SpannerInfo(spanner_id='age_par1', spanner_level=2, spanner_label=Html(text='age_par1'), spanner_units=N

# <span style="color:darkblue"> IV. Calculating Average Treatment Effect </span>


- Make sure to use robust standard errors

<font size = "5">

Effect of treatment on endline beliefs

In [13]:
reg_model = smf.ols("u_ave ~ treat ", dataset)
results = reg_model.fit(cov_type= "HC1")

mt.ETable([results], model_heads=["Treatment effect"]).make(type="gt")

GT(_tbl_data=  __index_level_0__ __index_level_1__                       0
0              coef   treat=Treatment  -7.422*** <br> (0.499)
1              coef         Intercept  63.563*** <br> (0.343)
2             stats      Observations                   5,244
3             stats                R²                   0.041, _body=<great_tables._gt_data.Body object at 0x35e276e00>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x35e632650>, _spanners=Spanners([SpannerInfo(spanner_id='u_ave', spanner_level=2, spanner_label=Html(text='u_ave'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='Treatment effect', spanner_level=1, spanner_label=Html(text='Treatment effect'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x35e2778a0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x35e277680>, _source_notes=[Html(text='Significance levels: * p &lt; 0.1, ** p &lt; 0.05, *** p &lt; 0.01. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x35e2863f0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x35dc27050>, <great_tables._gt_data.FormatInfo object at 0x35dc25310>, <great_tables._gt_data.FormatInfo object at 0x35dc24e90>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_includ

<font size = "5">

Add baseline covariates

In [14]:
reg_model = smf.ols("u_ave ~ treat + ave ", dataset)
results = reg_model.fit(cov_type= "HC1")

mt.ETable([results], model_heads=["Treatment + baseline control"]).make(type="gt")

GT(_tbl_data=  __index_level_0__ __index_level_1__                       0
0              coef   treat=Treatment  -7.056*** <br> (0.432)
1              coef               ave   0.508*** <br> (0.013)
2              coef         Intercept  39.655*** <br> (0.680)
3             stats      Observations                   5,244
4             stats                R²                   0.273, _body=<great_tables._gt_data.Body object at 0x31a2be300>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x31a2be4e0>, _spanners=Spanners([SpannerInfo(spanner_id='u_ave', spanner_level=2, spanner_label=Html(text='u_ave'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='Treatment + baseline control', spanner_level=1, spanner_label=Html(text='Treatment + baseline control'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x35e632b50>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x35e632750>, _source_notes=[Html(text='Significance levels: * p &lt; 0.1, ** p &lt; 0.05, *** p &lt; 0.01. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x35e287950>, _formats=[<great_tables._gt_data.FormatInfo object at 0x35d569010>, <great_tables._gt_data.FormatInfo object at 0x35d568710>, <great_tables._gt_data.FormatInfo object at 0x35d569d90>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(s

<font size = "5">

Test for heterogeneous effects


In [15]:
reg_model = smf.ols("u_ave ~ treat + ave + treat*ave", dataset)
results = reg_model.fit(cov_type= "HC1")

mt.ETable([results], model_heads=["Heterogeneous effect"]).make(type="gt")

GT(_tbl_data=  __index_level_0__      __index_level_1__                        0
0              coef        treat=Treatment  -25.998*** <br> (1.238)
1              coef                    ave    0.305*** <br> (0.018)
2              coef  treat=Treatment × ave    0.405*** <br> (0.024)
3              coef              Intercept   49.188*** <br> (0.884)
4             stats           Observations                    5,244
5             stats                     R²                    0.309, _body=<great_tables._gt_data.Body object at 0x31a38edd0>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x35e238600>, _spanners=Spanners([SpannerInfo(spanner_id='u_ave', spanner_level=2, spanner_label=Html(text='u_ave'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='Heterogeneous effect', spanner_level=1, spanner_label=Html(text='Heterogeneous effect'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x31a2bf7a0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x31a2bff20>, _source_notes=[Html(text='Significance levels: * p &lt; 0.1, ** p &lt; 0.05, *** p &lt; 0.01. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x31a404830>, _formats=[<great_tables._gt_data.FormatInfo object at 0x35e235010>, <great_tables._gt_data.FormatInfo object at 0x35e234410>, <great_tables._gt_data.FormatInfo object at 0x35e2d4d70>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss

<font size = "5">

Try it yourself!

Test for heterogeneous effects using other baseline covariates!

In [16]:
# Write your own code


# Test heterogeneous effects using other baseline covariates

# Literacy
reg_model = smf.ols("u_ave ~ treat + lit + treat*lit", data=dataset)
results_lit = reg_model.fit(cov_type="HC1")

# Parent age
reg_model = smf.ols("u_ave ~ treat + age_par1 + treat*age_par1", data=dataset)
results_age = reg_model.fit(cov_type="HC1")

# Household size
reg_model = smf.ols("u_ave ~ treat + tot_kids + treat*tot_kids", data=dataset)
results_kids = reg_model.fit(cov_type="HC1")

# Student age
reg_model = smf.ols("u_ave ~ treat + age + treat*age", data=dataset)
results_student_age = reg_model.fit(cov_type="HC1")

# Display the regression results
mt.ETable(
    [results_lit, results_age, results_kids, results_student_age],
    model_heads=["Literacy", "Parent age", "Household size", "Student age"]
).make(type="gt")






GT(_tbl_data=   __index_level_0__           __index_level_1__                       0  \
0               coef             treat=Treatment  -8.598*** <br> (0.931)   
1               coef                         lit      1.267 <br> (0.775)   
2               coef       treat=Treatment × lit      1.674 <br> (1.101)   
3               coef                    age_par1                           
4               coef  treat=Treatment × age_par1                           
5               coef                    tot_kids                           
6               coef  treat=Treatment × tot_kids                           
7               coef                         age                           
8               coef       treat=Treatment × age                           
9               coef                   Intercept  62.743*** <br> (0.667)   
10             stats                Observations                   5,220   
11             stats                          R²                   0.044   

                          1                       2                       3  
0   -10.344*** <br> (2.045)  -6.269*** <br> (1.540)     -0.017 <br> (2.302)  
1                                                                            
2                                                                            
3       -0.025 <br> (0.034)                                                  
4        0.074 <br> (0.049)                                                  
5                                0.178 <br> (0.190)                          
6                               -0.223 <br> (0.285)                          
7                                                     0.942*** <br> (0.127)  
8                                                    -0.653*** <br> (0.191)  
9    64.700*** <br> (1.387)  62.644*** <br> (0.994)  52.750*** <br> (1.546)  
10                    4,790                   5,244                   5,001  
11                     0.04                   0.041                   0.054  , _body=<great_tables._gt_data.Body object at 0x35dcd5790>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(2)'), column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(3)'), column_align='center', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(4)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x35e5089f0>, _spanners=Spanners([SpannerInfo(spanner_id='u_ave', spanner_level=2, spanner_label=Html(text='u_ave'), spanner_units=None, spanner_pattern=None, vars=['0', '1', '2', '3'], built=None), SpannerInfo(spanner_id='Literacy', spanner_level=1, spanner_label=Html(text='Literacy'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='Parent age', spanner_level=1, spanner_label=Html(text='Parent age'), spanner_units=None, spanner_pattern=None, vars=['1'], built=None), SpannerInfo(spanner_id='Household size', spanner_level=1, spanner_label=Html(text='Household size'), spanner_units=None, spanner_pattern=None, vars=['2'], built=None), SpannerInfo(spanner_id='Student age', spanner_level=1, spanner_label=Html(text='Student age'), spanner_units=None, spanner_pattern=None, vars=['3'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x31a38dd30>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x31a3